In [28]:
import pandas as pd
import numpy as np
df = pd.read_csv("/content/DateFruit_Dataset.csv")
df.head()
df.isnull()

x = df.drop("Class", axis = 1)
y = df["Class"]
df["Class"].unique()

from sklearn.preprocessing import LabelEncoder, StandardScaler
le = LabelEncoder()
y = le.fit_transform(y)
scaler = StandardScaler()

from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test = train_test_split(x, y, test_size=0.2, random_state=42)
x_train_scaled = scaler.fit_transform(x_train)
x_test_scaled = scaler.transform(x_test)


#ann model // creation
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

x_train_tensor = torch.tensor(x_train_scaled, dtype = torch.float32)
y_train_tensor = torch.tensor(y_train, dtype = torch.long)
x_test_tensor = torch.tensor(x_test_scaled, dtype = torch.float32)
y_test_tensor = torch.tensor(y_test, dtype = torch.long)
train_dataset = TensorDataset(x_train_tensor,y_train_tensor)
test_dataset = TensorDataset(x_test_tensor,y_test_tensor)
train_loader = DataLoader(train_dataset,batch_size=32,shuffle=True)
test_loader = DataLoader(test_dataset,batch_size=32)

#building our model
class ANN(nn.Module):
  def __init__(self):
    super(ANN,self).__init__()
    self.model = nn.Sequential(
        #1st hidden layer
        nn.Linear(x.shape[1],64),
        nn.ReLU(),
        #2nd hidden layer
        nn.Linear(64,64),
        nn.ReLU(),
        #output layer
        nn.Linear(64,7))

  def forward(self,x):
    return self.model(x)


#loss and optimizers
model = ANN()
criteria = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())


#Training the NN
epochs = 100
for epoch in range(epochs):
  model.train()
  running_loss = 0.0
  for xb,yb in train_loader:
    optimizer.zero_grad()
    outputs = model(xb)
    loss = criteria(outputs,yb)
    loss.backward()
    optimizer.step() #parameter updation happens here
    running_loss += loss.item()
  train_loss = running_loss / len(train_loader)
  print(f"epoch = {epoch+1}/{epochs}, loss = {train_loss}")

#model evaluation
model.eval()
total = 0
correct = 0
with torch.no_grad():
  for xb,yb in test_loader:
    outputs = model(xb)
    _,predicted = torch.max(outputs,1)
    correct += (predicted == yb).sum().item()
    total += yb.size(0)
print("accuracy:",correct/total*100)

epoch = 1/100, loss = 1.7100316026936406
epoch = 2/100, loss = 1.0351266316745593
epoch = 3/100, loss = 0.659852543602819
epoch = 4/100, loss = 0.5075575553852579
epoch = 5/100, loss = 0.4293916484583979
epoch = 6/100, loss = 0.37759137607139087
epoch = 7/100, loss = 0.3263860889103102
epoch = 8/100, loss = 0.2941440253154091
epoch = 9/100, loss = 0.27296673538892163
epoch = 10/100, loss = 0.25995089761588885
epoch = 11/100, loss = 0.24551525388074957
epoch = 12/100, loss = 0.22233192298723303
epoch = 13/100, loss = 0.20481164721043213
epoch = 14/100, loss = 0.2063524518971858
epoch = 15/100, loss = 0.1901872724942539
epoch = 16/100, loss = 0.18131836860076242
epoch = 17/100, loss = 0.17011315521338713
epoch = 18/100, loss = 0.16323150937323985
epoch = 19/100, loss = 0.1573417811937954
epoch = 20/100, loss = 0.1486817906084268
epoch = 21/100, loss = 0.14411452930906546
epoch = 22/100, loss = 0.14326888160861057
epoch = 23/100, loss = 0.13547003131521784
epoch = 24/100, loss = 0.1270692

In [ ]:
df